In [111]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [112]:
import pandas as pd
data=pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
data=data.head(1000)

In [113]:
import re
def remove_tag(text):
    clean_text=re.sub(r'<[^>]+>', '', text)
    return clean_text.lower()

In [114]:
data['review']=data['review'].apply(remove_tag)
data

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
995,nothing is sacred. just ask ernie fosselius. t...,positive
996,i hated it. i hate self-aware pretentious inan...,negative
997,i usually try to be professional and construct...,negative
998,if you like me is going to see this in a film ...,negative


In [115]:
def label_conv(label):
    if label=='positive':
        return 1
    else:
        return 0

data['sentiment']=data['sentiment'].apply(label_conv)
data

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. the filming tec...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1
...,...,...
995,nothing is sacred. just ask ernie fosselius. t...,1
996,i hated it. i hate self-aware pretentious inan...,0
997,i usually try to be professional and construct...,0
998,if you like me is going to see this in a film ...,0


In [116]:
from sklearn.model_selection import train_test_split

x=data['review']
y=data['sentiment']
xtrain,xtext,ytrain,ytest=train_test_split(x,y,test_size=0.2)

In [117]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [118]:
train=xtrain.tolist()

In [119]:
train_encodings = tokenizer(train, truncation=True, padding=True,max_length=128,return_tensors="pt")
train_encodings

{'input_ids': tensor([[ 101, 1005, 2162,  ..., 8321, 1012,  102],
        [ 101, 2023, 2143,  ..., 2291, 1012,  102],
        [ 101, 3100, 2023,  ..., 5236, 6925,  102],
        ...,
        [ 101, 2023, 2537,  ...,    0,    0,    0],
        [ 101, 2017, 4929,  ..., 1999, 1996,  102],
        [ 101, 1045, 4149,  ..., 1011, 9725,  102]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])}

In [120]:
import torch

class YourDatasetClass(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        # FIX 1: Convert pandas Series directly to a numpy array or list
        # This completely strips out the problematic pandas indices
        self.labels = labels.to_numpy() if hasattr(labels, 'to_numpy') else list(labels)

    def __getitem__(self, idx):
        # FIX 2: Added .clone().detach() to resolve your PyTorch UserWarning
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        
        # This will now correctly look up by numerical sequence index
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = YourDatasetClass(train_encodings, ytrain)

In [121]:
train_dataset

In [122]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=1,              # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=64,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
)

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

trainer = Trainer(
    model=model,                         # the instantiated ðŸ¤— Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset         # training dataset
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(load

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=50, training_loss=0.6898047637939453, metrics={'train_runtime': 295.0014, 'train_samples_per_second': 2.712, 'train_steps_per_second': 0.169, 'total_flos': 26493479731200.0, 'train_loss': 0.6898047637939453, 'epoch': 1.0})

In [123]:
from transformers import pipeline

# 1. Initialize the text classification pipeline with your trained model and tokenizer
classifier = pipeline(
    "text-classification", 
    model=model,          # Your trained model variable
    tokenizer=tokenizer   # Your tokenizer variable
)

# 2. Predict on new sentences
new_sentences = [
    "I absolutely loved this movie, the acting was superb!",
    "What a waste of time. The plot made absolutely no sense."
]

predictions = classifier(new_sentences)

# 3. Print the results
for text, pred in zip(new_sentences, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred['label']} (Confidence: {pred['score']:.4f})\n")


Text: I absolutely loved this movie, the acting was superb!
Prediction: LABEL_1 (Confidence: 0.5210)

Text: What a waste of time. The plot made absolutely no sense.
Prediction: LABEL_0 (Confidence: 0.5160)

